<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/es/code/stable-baselines/cartpole-sb3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DQN y PPO para _cartpole_ con Stable-Baselines3

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
Este trabajo está bajo licencia [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Basado en:
> A. Raffin et al. (2021) Stable-Baselines3: Reliable Reinforcement Learning Implementations. _Journal of Machine Learning Research_ 22(268):1-8. Disponible en [GitHub](https://github.com/DLR-RM/stable-baselines3).

> A. Raffin (2020) RL Baselines3 Zoo. Disponible en [GitHub](https://github.com/DLR-RM/rl-baselines3-zoo).

En los cuadernos de la carpeta [`from-scratch`](https://github.com/jgromero/drl-csic/tree/es/code/from-scratch) implementamos nosotros mismos DQN y REINFORCE. Aquí resolvemos el mismo problema con [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) (SB3), una biblioteca de implementaciones fiables y probadas de los algoritmos de aprendizaje por refuerzo profundo (DRL) más comunes, construida sobre PyTorch. Con SB3, el agente, la red neuronal, la memoria de experiencias (_replay buffer_) y el bucle de entrenamiento ya están implementados: solo tenemos que elegir un algoritmo, fijar sus hiperparámetros y llamar a `learn()`.

Entrenaremos dos agentes:

*   [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html): el mismo algoritmo basado en valor de [cartpole-dqn.ipynb](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/cartpole-dqn.ipynb).
*   [PPO](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html): un algoritmo de gradiente de política (actor-crítico) que mejora las ideas de REINFORCE ([cartpole-reinforce.ipynb](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/cartpole-reinforce.ipynb)).

## Configuración

In [ ]:
%pip install "gymnasium[classic-control]>=1.1" "stable-baselines3>=2.6" tqdm rich

In [ ]:
# modo de renderizado 'rgb_array' (cambiar a 'human' para ejecución local con pantalla)
render_mode = "rgb_array"

## Entorno

Usaremos [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), un problema de control sencillo.

<img src="https://github.com/jgromero/drl-csic/blob/es/img/cartpole.gif?raw=true"/>

El entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) se considera resuelto cuando el agente obtiene una puntuación media de al menos $475$ puntos en 100 episodios consecutivos. Los episodios se truncan a los $500$ pasos, que es por tanto la puntuación máxima.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

env = gym.make("CartPole-v1", render_mode = render_mode)
print(env.observation_space)
print(env.action_space)

SB3 necesita que el entorno esté envuelto en un [`Monitor`](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) para registrar la puntuación y la duración de cada episodio. Usaremos estos valores para representar las curvas de aprendizaje.

In [ ]:
SOLVED_SCORE = 475.0   # puntuación media (últimos 100 episodios) para considerar resuelto CartPole-v1

def plot_scores(env, title):
    """Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos 100.

    Parámetros
    ==========
        env: entorno (o entorno vectorizado) envuelto con Monitor
        title (str): título de la gráfica
    """
    # un entorno vectorizado contiene varias copias del entorno envueltas con Monitor
    monitors = [env] if isinstance(env, Monitor) else env.envs
    scores = np.concatenate([m.get_episode_rewards() for m in monitors])
    ends = np.concatenate([np.cumsum(m.get_episode_lengths()) for m in monitors])
    scores = scores[np.argsort(ends)]   # ordenar los episodios por el paso de tiempo (de cada entorno) en que terminaron

    avg = [np.mean(scores[max(0, i-99):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label='Puntuación')
    plt.plot(np.arange(len(scores)), avg, label='Puntuación media (últimos 100)')
    plt.axhline(SOLVED_SCORE, color='gray', linestyle='--', label='Resuelto')
    plt.ylabel('Puntuación')
    plt.xlabel('Episodio nº')
    plt.title(title)
    plt.legend()
    plt.show()

## Algoritmo DQN

La clase [`DQN`](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) incluye la red Q, la red objetivo (_target network_), la memoria de experiencias (_replay buffer_) y la estrategia de exploración $\epsilon$-_greedy_. Comparemos sus parámetros con los de nuestra implementación en [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/dqn_agent.py):

| Parámetro de SB3 | Significado | `dqn_agent.py` |
|---|---|---|
| `buffer_size` | tamaño de la memoria de experiencias | `BUFFER_SIZE` |
| `batch_size` | tamaño del minilote | `BATCH_SIZE` |
| `gamma` | factor de descuento | `GAMMA` |
| `learning_rate` | tasa de aprendizaje | `LR` |
| `train_freq` | cada cuántos pasos se actualiza la red | `UPDATE_EVERY` |
| `target_update_interval` | cada cuántos pasos se actualiza la red objetivo | `TAU` (actualización suave) |
| `exploration_*` | calendario de $\epsilon$-_greedy_ | `eps_start`, `eps_end`, `eps_decay` |
| `policy_kwargs` | arquitectura de la red Q | [model.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/model.py) |

Usamos los hiperparámetros ajustados para CartPole-v1 en el [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/dqn.yml). En lugar de un número de episodios, SB3 entrena durante un número dado de pasos de tiempo (`total_timesteps`).

In [ ]:
env_dqn = Monitor(gym.make("CartPole-v1"))

model_dqn = DQN(
    "MlpPolicy",                        # red Q: perceptrón multicapa
    env_dqn,
    learning_rate=2.3e-3,
    batch_size=64,
    buffer_size=100_000,
    learning_starts=1000,               # pasos con acciones aleatorias antes de empezar a aprender
    gamma=0.99,
    target_update_interval=10,
    train_freq=256,                     # actualizar la red cada 256 pasos...
    gradient_steps=128,                 # ...con 128 minilotes
    exploration_fraction=0.16,          # epsilon decrece linealmente durante el primer 16% del entrenamiento...
    exploration_final_eps=0.04,         # ...hasta este valor
    policy_kwargs=dict(net_arch=[256, 256]),
    seed=0,
    verbose=0,
)

model_dqn.learn(total_timesteps=50_000, progress_bar=True)
model_dqn.save("cartpole-sb3-dqn")   # guardar el agente entrenado

plot_scores(env_dqn, 'DQN')

La puntuación mostrada durante el entrenamiento se obtiene con la política $\epsilon$-_greedy_, es decir, con algunas acciones aleatorias. Para evaluar la política aprendida, evaluamos el agente con la política _greedy_ (`deterministic=True`) durante 100 episodios.

In [ ]:
mean_score, std_score = evaluate_policy(model_dqn, Monitor(gym.make("CartPole-v1")), n_eval_episodes=100, deterministic=True)
print('DQN\tPuntuación media (100 episodios): {:.2f} +/- {:.2f}'.format(mean_score, std_score))

## Algoritmo PPO

[Proximal Policy Optimization](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html) (PPO) es un método de gradiente de política, como REINFORCE, con varias mejoras:

*   Es un método _actor-critic_ (actor-crítico): además de la red de política (actor), aprende una red de valor (crítico) que se usa para reducir la varianza de la estimación del gradiente (en lugar del retorno normalizado $G_t$ que usamos en REINFORCE).
*   Limita (_clipping_) el tamaño de cada actualización de la política, de modo que la nueva política no se aleje demasiado de la anterior. Esto permite reutilizar cada lote de experiencia para varios pasos de gradiente (`n_epochs`).
*   Recoge experiencia de varias copias del entorno en paralelo (`n_envs`), que SB3 construye con [`make_vec_env`](https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html).

De nuevo usamos los hiperparámetros del [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/ppo.yml). Nótese que solo necesitamos cambiar la clase del modelo: el resto del código es el mismo.

In [ ]:
env_ppo = make_vec_env("CartPole-v1", n_envs=8, seed=0)   # 8 entornos en paralelo, cada uno envuelto con Monitor

model_ppo = PPO(
    "MlpPolicy",                        # redes de actor y crítico: perceptrones multicapa
    env_ppo,
    learning_rate=1e-3,
    n_steps=32,                         # pasos recogidos de cada entorno antes de cada actualización
    batch_size=256,
    n_epochs=20,                        # pasos de gradiente (épocas) con cada lote de experiencia
    gamma=0.98,
    gae_lambda=0.8,
    clip_range=0.2,                     # cambio máximo de la política en cada actualización
    ent_coef=0.0,
    seed=0,
    verbose=0,
)

model_ppo.learn(total_timesteps=100_000, progress_bar=True)
model_ppo.save("cartpole-sb3-ppo")   # guardar el agente entrenado

plot_scores(env_ppo, 'PPO')

In [ ]:
mean_score, std_score = evaluate_policy(model_ppo, Monitor(gym.make("CartPole-v1")), n_eval_episodes=100, deterministic=True)
print('PPO\tPuntuación media (100 episodios): {:.2f} +/- {:.2f}'.format(mean_score, std_score))

## Visualizar el agente entrenado

Posteriormente podemos cargar los agentes entrenados y visualizar cómo actúan en el entorno. (Se requiere ejecución local para el renderizado 2D, no Google Colaboratory.)

In [ ]:
# cargar el agente entrenado desde `cartpole-sb3-ppo.zip` (usar DQN.load("cartpole-sb3-dqn") para el agente DQN)
model_load = PPO.load("cartpole-sb3-ppo")

for i in range(3):
    state, _ = env.reset()
    for j in range(500):
        action, _ = model_load.predict(state, deterministic=True)
        env.render()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break

env.close()